In [1]:
from sklearn.datasets import fetch_california_housing, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor


def load_and_split(X, y, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


def fetch_openml_numeric(name, target):
    ds = fetch_openml(name, as_frame=True)
    X = ds.data
    y = ds.target.astype(float)

    # оставляем только числовые признаки
    X = X.select_dtypes(include=[np.number])

    return X.values, y.values

def fetch_openml_numeric_by_id(data_id):
    ds = fetch_openml(data_id=data_id, as_frame=True)
    X = ds.data.select_dtypes(include=[np.number])
    y = ds.target.astype(float)
    return X.values, y.values


In [2]:
# 4. Facebook Comment Volume
# ~50k samples
X, y = fetch_openml_numeric_by_id(4549)
print(X.shape)
X = X[:300000, :]
y = y[:300000]

def load_and_split_2(X, y, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=random_state
    )
    X_linreg, X_test, y_linreg, y_test = train_test_split(
        X_test, y_test, test_size=0.5, random_state=random_state
    )
    return X_train, X_val, X_linreg, X_test, y_train, y_val, y_linreg, y_test

name = 'facebook_comment_volume'
value = load_and_split(X, y)
del X, y

(583250, 77)


In [3]:
X_train, X_val, X_test, y_train, y_val, y_test = value
del value

cb = CatBoostRegressor(
    iterations=10_000,
    learning_rate=1e-2,
    loss_function="RMSE",
    early_stopping_rounds=50,
    use_best_model=True,
    verbose=False,
)

cb.fit(
    X_train, y_train,
    eval_set=(X_val, y_val)
)

n_trees = cb.tree_count_
print(f"Trained trees: {n_trees}")


Trained trees: 600


In [4]:
from tqdm import tqdm

def tree_predictions(model, X):
    T = model.tree_count_
    preds = np.zeros((X.shape[0], T), dtype=np.float32)

    for t in tqdm(range(T)):
        preds[:, t] = model.predict(
            X,
            ntree_start=t,
            ntree_end=t + 1,
            prediction_type="RawFormulaVal"
        )
    return preds

Z_train = tree_predictions(cb, X_train)
Z_test = tree_predictions(cb, X_test)

100%|██████████| 600/600 [08:35<00:00,  1.16it/s]


In [5]:
from sklearn.metrics import r2_score
r2_score(y_test, Z_test.sum(axis=1))

0.7819954537994142

In [6]:
from sklearn.linear_model import lasso_path
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler(with_mean=True, with_std=True)
Zs = scaler.fit_transform(Z_train)

alphas, coefs, _ = lasso_path(
    Zs,
    y_train,
    max_iter=50000
)

def tree_order_from_lasso_path(coefs):
    order = []
    active = set()

    for k in range(coefs.shape[1]):
        nz = set(np.nonzero(coefs[:, k])[0])
        new = nz - active
        if new:
            order.extend(sorted(new))
            active |= new
    return order

tree_order = tree_order_from_lasso_path(coefs)

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

Ks = [1, 3, 5, 10, 20, 30, 50, 70, 100, 150, 250, 600]

results = []

for K in Ks:
    idx = tree_order[:K]

    lr = LinearRegression(fit_intercept=False)
    lr.fit(Z_train[:, idx], y_train)

    y_pred = lr.predict(Z_test[:, idx])
    score = r2_score(y_test, y_pred)

    results.append({
        "method": "lasso_pruned",
        "K": K,
        "r2_test": score
    })

y_cb = cb.predict(X_test)
cb_score = r2_score(y_test, y_cb)

results.append({
    "method": "catboost_full",
    "K": n_trees,
    "r2_test": cb_score
})


results

[{'method': 'lasso_pruned', 'K': 1, 'r2_test': 0.6352638071257086},
 {'method': 'lasso_pruned', 'K': 3, 'r2_test': 0.6625862257789896},
 {'method': 'lasso_pruned', 'K': 5, 'r2_test': 0.6740008430503193},
 {'method': 'lasso_pruned', 'K': 10, 'r2_test': 0.677977402513404},
 {'method': 'lasso_pruned', 'K': 20, 'r2_test': 0.690775957439905},
 {'method': 'lasso_pruned', 'K': 30, 'r2_test': 0.6962624929729132},
 {'method': 'lasso_pruned', 'K': 50, 'r2_test': 0.7014318402443973},
 {'method': 'lasso_pruned', 'K': 70, 'r2_test': 0.7044907219785572},
 {'method': 'lasso_pruned', 'K': 100, 'r2_test': 0.7043624054852202},
 {'method': 'lasso_pruned', 'K': 150, 'r2_test': 0.6853307173959388},
 {'method': 'lasso_pruned', 'K': 250, 'r2_test': 0.7719164044332786},
 {'method': 'lasso_pruned', 'K': 600, 'r2_test': 0.77244174421062},
 {'method': 'catboost_full', 'K': 600, 'r2_test': 0.7819954510905907}]

In [8]:
[{'method': 'lasso_pruned', 'K': 1, 'r2_test': 0.8373943861364928},
 {'method': 'lasso_pruned', 'K': 3, 'r2_test': 0.8654146445316396},
 {'method': 'lasso_pruned', 'K': 5, 'r2_test': 0.8718546969358935},
 {'method': 'lasso_pruned', 'K': 10, 'r2_test': 0.880131855914774},
 {'method': 'lasso_pruned', 'K': 20, 'r2_test': 0.8825604447403103},
 {'method': 'lasso_pruned', 'K': 30, 'r2_test': 0.8720554987761011},
 {'method': 'lasso_pruned', 'K': 50, 'r2_test': 0.8640613769496706},
 {'method': 'lasso_pruned', 'K': 70, 'r2_test': 0.8458298741763308},
 {'method': 'lasso_pruned', 'K': 100, 'r2_test': 0.8109551498020799},
 {'method': 'lasso_pruned', 'K': 150, 'r2_test': 0.8078933667911017},
 {'method': 'lasso_pruned', 'K': 250, 'r2_test': 0.7963834181491194},
 {'method': 'lasso_pruned', 'K': 788, 'r2_test': 0.7751020321812518},
 {'method': 'catboost_full', 'K': 788, 'r2_test': 0.8900318365122302}]

[{'method': 'lasso_pruned', 'K': 1, 'r2_test': 0.8373943861364928},
 {'method': 'lasso_pruned', 'K': 3, 'r2_test': 0.8654146445316396},
 {'method': 'lasso_pruned', 'K': 5, 'r2_test': 0.8718546969358935},
 {'method': 'lasso_pruned', 'K': 10, 'r2_test': 0.880131855914774},
 {'method': 'lasso_pruned', 'K': 20, 'r2_test': 0.8825604447403103},
 {'method': 'lasso_pruned', 'K': 30, 'r2_test': 0.8720554987761011},
 {'method': 'lasso_pruned', 'K': 50, 'r2_test': 0.8640613769496706},
 {'method': 'lasso_pruned', 'K': 70, 'r2_test': 0.8458298741763308},
 {'method': 'lasso_pruned', 'K': 100, 'r2_test': 0.8109551498020799},
 {'method': 'lasso_pruned', 'K': 150, 'r2_test': 0.8078933667911017},
 {'method': 'lasso_pruned', 'K': 250, 'r2_test': 0.7963834181491194},
 {'method': 'lasso_pruned', 'K': 788, 'r2_test': 0.7751020321812518},
 {'method': 'catboost_full', 'K': 788, 'r2_test': 0.8900318365122302}]